# Waste Detection 2-Stage Pipeline (Final)

Notebook thuc thi toan bo quy trinh tren Kaggle.

### Kien truc:
1. **Stage 1:** YOLO26s phat hien vung rac (binary), `conf=0.40`
2. **Stage 2:** EfficientNet-B2 phan loai 6 lop (5 rac + Background loc FP)

### Cau hinh:
| Thanh phan | Gia tri |
|---|---|
| Stage 1 model | YOLO26s (100 epochs) |
| Stage 1 conf | **0.40** |
| Stage 2 model | EfficientNet-B2 |
| Stage 2 Dropout | 0.5 |
| Mixup alpha | 0.3 |
| TTA | 5 augments |
| Macro F1 | Tinh tren 5 lop rac (khong tinh Background) |
| Model Stage 1 | models/final_best.pt |
| Model Stage 2 | models/final_best.pth |

**Yeu cau:** Import file len Kaggle, bat internet va GPU T4 x2 de toi uu toc do train.

In [ ]:
# ============================================================
# 1. Tai Ma nguon & Cai dat thu vien
# ============================================================
!git clone https://github.com/Shiba-dotcom/waste-detection2-Stage.git
!pip install -q ultralytics timm

In [ ]:
# ============================================================
# 2. Nap Du lieu Ngoai lai (TACO, TrashNet, RealWaste)
# ============================================================
import os, shutil

!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/TrashNet
!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/RealWaste
!mkdir -p /kaggle/working/waste-detection2-Stage/data/raw

datasets_to_copy = [
    {"src": "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/TrashNet"},
    {"src": "/kaggle/input/datasets/sohamchaudhari2004/taco-trash-detection-dataset/data",
     "dst": "/kaggle/working/waste-detection2-Stage/data/raw"},
    {"src": "/kaggle/input/datasets/joebeachcapital/realwaste/realwaste-main/RealWaste",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/RealWaste"}
]

for task in datasets_to_copy:
    if os.path.exists(task["src"]):
        os.makedirs(task["dst"], exist_ok=True)
        shutil.copytree(task["src"], task["dst"], dirs_exist_ok=True)
        print(f"Da tai: {os.path.basename(task['src'])}")
    else:
        print(f"Bo qua: {task['src']} (Khong tim thay tren Kaggle Dataset)")

In [ ]:
# ============================================================
# 3. Tien xu ly Du lieu (Data Pipeline)
# ============================================================
%cd /kaggle/working/waste-detection2-Stage

print("--- 3.1 Don dep & Tao nhan YOLO ---")
!python src/data_prep/data_cleaning.py
!python src/Training_dataYolo.py
!python src/data_prep/split_dataset.py

print("--- 3.2 Chuan bi du lieu cho Stage 2 (Classifier) ---")
!python src/data_prep/crop_for_classification.py
!python src/data_prep/merge_external_datasets.py
!python src/data_prep/generate_background.py

print("Hoan tat chuan bi du lieu!")

In [ ]:
# ============================================================
# 4. Huan luyen Stage 1 (YOLO26s - Binary Detector)
# Bo qua neu da co models/final_best.pt
# Script: src/train_stage1_detector.py
#   - Model  : yolo26s.pt
#   - Epochs : 100 (patience=20)
#   - Output : models/final_best.pt
# ============================================================
from pathlib import Path

if not Path("models/final_best.pt").exists():
    !python src/train_stage1_detector.py
else:
    print("Da tim thay models/final_best.pt, bo qua train Stage 1.")


In [ ]:
# ============================================================
# 5. Huan luyen Stage 2 (EfficientNet-B2 - 6 Lop Classifier)
# Dropout=0.5 | WeightDecay=1e-4 | Mixup(alpha=0.3) | TTA
# Luu ra: models/final_best.pth
# ============================================================
!python src/train_stage2_classifier.py